In [5]:
import sys
from types import ModuleType

# 제거된 vertexai 모듈을 메모리에 가짜로 등록
dummy_chat_vertex = ModuleType("langchain_community.chat_models.vertexai")
dummy_chat_vertex.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = dummy_chat_vertex

dummy_llm_vertex = ModuleType("langchain_community.llms.vertexai")
dummy_llm_vertex.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = dummy_llm_vertex

In [6]:
import pandas as pd
from ragas import EvaluationDataset

# 검색 실패 분석을 위해 의도적으로 품질 결함이 포함된 샘플 데이터를 정의합니다.
# 1번 케이스: 메타데이터 필터 미흡으로 과거(2023년) 문서가 검색된 사례
# 2번 케이스: 청킹 분할 실패로 운영 시간 문장이 중간에 잘려 나간 사례
# 3번 케이스: 정상적으로 검색된 성공 사례
eval_raw_data = {
    "user_input": [
        "2024년 1학기 교내 장학금 신청 마감일은 언제인가요?",
        "중앙도서관 주말 운영 시간 및 대출 규정은 어떻게 되나요?",
        "기숙사 식당 아침 식사 시간은 언제인가요?"
    ],
    "reference": [
        "2024년 1학기 교내 장학금 신청 마감일은 2024년 5월 20일 18시까지입니다.",
        "중앙도서관 주말 운영 시간은 오전 10시부터 오후 5시까지이며, 1인당 최대 5권까지 대출 가능합니다.",
        "기숙사 식당 조식 운영 시간은 오전 7시부터 오전 9시까지입니다."
    ],
    "retrieved_contexts": [
        [
            "2023년 2학기 교내 장학금 신청 마감일은 11월 30일까지입니다.",
            "장학재단 외부 장학금 접수 안내 공지사항입니다."
        ],
        [
            "중앙도서관 주말 운영 안내: 토요일과 일요일은 오전 10시부터 오후 5시까지... (문맥 끊김)",
            "도서관 매점은 평일 오전 9시부터 오후 6시까지 운영됩니다."
        ],
        [
            "기숙사 식당 운영 안내: 조식은 07:00~09:00, 중식은 11:30~13:30에 제공됩니다.",
            "기숙사 헬스장은 24시간 자율 개방됩니다."
        ]
    ]
}

# 딕셔너리 데이터를 판다스 DataFrame으로 변환합니다.
df_search_eval = pd.DataFrame(eval_raw_data)

# RAGAS표준 EvaluationDataset 객체로 변환합니다.
ragas_dataset = EvaluationDataset.from_pandas(df_search_eval)

print("=== [RAGAS 평가용 데이터셋 변환 완료] ===")
print(ragas_dataset)

=== [RAGAS 평가용 데이터셋 변환 완료] ===
EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference'], len=3)


In [7]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from ragas.metrics import ContextPrecision, ContextRecall
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

load_dotenv()

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="none",
    temperature=0
)

evaluator_llm = LangchainLLMWrapper(model, bypass_temperature=True)

metrics = [
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]

evaluation_result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics,
    raise_exceptions=True
)


C:\Users\yonggyo\AppData\Local\Temp\ipykernel_4136\2118014026.py:4: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\yonggyo\AppData\Local\Temp\ipykernel_4136\2118014026.py:4: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall
C:\Users\yonggyo\AppData\Local\Temp\ipykernel_4136\2118014026.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_k

In [10]:
df_result = evaluation_result.to_pandas()
print(df_result[['context_precision', 'context_recall']])

   context_precision  context_recall
0                0.0             0.0
1                1.0             0.0
2                1.0             1.0


In [12]:
SCORE_THRESHOLD = 0.8

failed_cases = df_result[(df_result['context_precision'] < SCORE_THRESHOLD) | (df_result['context_recall'] < SCORE_THRESHOLD)].copy()

failed_cases

,user_input,retrieved_contexts,reference,context_precision,context_recall
0,2024년 1학기 교내 장학금 신청 마감일은 언제인가요?,"[2023년 2학기 교내 장학금 신청 마감일은 11월 30일까지입니다., 장학재단 ...",2024년 1학기 교내 장학금 신청 마감일은 2024년 5월 20일 18시까지입니다.,0.0,0.0
1,중앙도서관 주말 운영 시간 및 대출 규정은 어떻게 되나요?,[중앙도서관 주말 운영 안내: 토요일과 일요일은 오전 10시부터 오후 5시까지......,"중앙도서관 주말 운영 시간은 오전 10시부터 오후 5시까지이며, 1인당 최대 5권까...",1.0,0.0


In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template(
    """
    당신은 RAG 검색 품질 문제를 진단하는 전문가 입니다.
    아래의 질문, 모범 정답, 실제 검색된 문서를 대조하여 검색 실패 원인을 분석하세요.

    [질문]: {question}
    [모범 정답]: {reference}
    [검색된 문서]
    {contexts}

    다음 
    1. 실패 유형: [청킹 실패 / 임베딩 실패 / 메타데이터 필터링 / 기타 실패 중 택 1]
    2. 상세 원인: [실패 이유를 2줄 이내로 명확하게 진술]
    3. 품질 개선안: [해결을 위한 구체적인 조치 사항을 2줄 이내로 명확하게 진술]
    """
)

chain = prompt | model | StrOutputParser()

reports = []
for _, row in failed_cases.iterrows():
    report = chain.invoke({
        "question": row["user_input"],
        "reference": row["reference"],
        "contexts": row['retrieved_contexts']
    })
    reports.append(report)

failed_cases["report"] = reports

output_csv_path = "search_failure_report.csv"
failed_cases.to_csv(output_csv_path, index=False, encoding="utf-8-sig")

failed_cases

,user_input,retrieved_contexts,reference,context_precision,context_recall,report
0,2024년 1학기 교내 장학금 신청 마감일은 언제인가요?,"[2023년 2학기 교내 장학금 신청 마감일은 11월 30일까지입니다., 장학재단 ...",2024년 1학기 교내 장학금 신청 마감일은 2024년 5월 20일 18시까지입니다.,0.0,0.0,1. 실패 유형: **기타 실패**\n2. 상세 원인: 검색 결과가 2023년 2학...
1,중앙도서관 주말 운영 시간 및 대출 규정은 어떻게 되나요?,[중앙도서관 주말 운영 안내: 토요일과 일요일은 오전 10시부터 오후 5시까지......,"중앙도서관 주말 운영 시간은 오전 10시부터 오후 5시까지이며, 1인당 최대 5권까...",1.0,0.0,1. 실패 유형: **청킹 실패**\n2. 상세 원인: 주말 운영 시간은 검색됐지만...
